## Bloque 1 – Carga de Datos
Cargamos el dataset `amazon_alexa.tsv`. Contiene reseñas reales de usuarios del producto **Amazon Alexa / Echo Dot**, con una columna `verified_reviews` (texto) y `feedback` (1=positivo, 0=negativo).

In [22]:
import pandas as pd
import numpy as np
import tensorflow as tf
import re
import nltk
from collections import Counter
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Descargar stopwords de NLTK (solo la primera vez)
nltk.download('stopwords', quiet=True)

# Cargar dataset
df = pd.read_csv("amazon_alexa.tsv", sep='\t')

reseñas  = df['verified_reviews'].astype(str).tolist()
etiquetas = df['feedback'].values

print(f"Total de reseñas cargadas : {len(reseñas)}")
print(f"Distribución de clases    : {dict(pd.Series(etiquetas).value_counts())}")
print("\nEjemplo de reseña positiva:")
print(df[df['feedback']==1]['verified_reviews'].iloc[0])
print("\nEjemplo de reseña negativa:")
print(df[df['feedback']==0]['verified_reviews'].iloc[0])

Total de reseñas cargadas : 3150
Distribución de clases    : {1: np.int64(2893), 0: np.int64(257)}

Ejemplo de reseña positiva:
Love my Echo!

Ejemplo de reseña negativa:
It's like Siri, in fact, Siri answers more accurately then Alexa.  I don't see a real need for it in my household, though it was a good bargain on prime day deals.


---
## Bloque 2 – Tokenización y Padding
Convertimos las frases en secuencias numéricas de longitud uniforme.

> **Nota importante**: En análisis de sentimiento **NO eliminamos stopwords** en este paso, porque palabras como "not", "never" o "no" aportan significado crucial al sentimiento.

In [23]:
vocab_size   = 1000   # Solo las 1000 palabras más frecuentes
max_length   = 50     # Longitud máxima de cada reseña (en palabras)
padding_type = 'post' # Rellenar con 0s al final si es corta
trunc_type   = 'post' # Cortar el final si es demasiado larga
oov_tok      = "<OOV>" # Token para palabras desconocidas

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(reseñas)  # Aprende el vocabulario

sequences = tokenizer.texts_to_sequences(reseñas)  # Convierte a números
padded    = pad_sequences(sequences, maxlen=max_length,
                          padding=padding_type, truncating=trunc_type)

print(f"Forma de la matriz de entrada: {padded.shape}")
print(f"Ejemplo de secuencia (reseña 0): {padded[0][:20]} ...")

Forma de la matriz de entrada: (3150, 50)
Ejemplo de secuencia (reseña 0): [11  8 12  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0] ...


---
## Bloque 3 – Modelo con Bidirectional LSTM
Construimos la red neuronal con:
- **Embedding**: Aprende representaciones vectoriales de cada palabra
- **Bidirectional LSTM**: Lee la frase en ambos sentidos (izquierda→derecha y derecha→izquierda) para capturar mejor el contexto
- **Dense + Sigmoid**: Clasifica la salida entre 0 (negativo) y 1 (positivo)

In [24]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 16, input_length=max_length),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/home/ciabd14/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Entrenamiento
Usamos el 80% de los datos para entrenar y el 20% restante para validar que el modelo generaliza bien.

In [25]:
history = model.fit(
    padded, etiquetas,
    epochs=10,
    validation_split=0.2,
    verbose=1
)

# Mostrar precisión final en validación
val_acc = history.history['val_accuracy'][-1]
print(f"\nModelo entrenado | Precisión en validación: {val_acc*100:.2f}%")

Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9040 - loss: 0.3518 - val_accuracy: 0.9317 - val_loss: 0.2415
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9151 - loss: 0.2722 - val_accuracy: 0.9317 - val_loss: 0.2256
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9202 - loss: 0.2376 - val_accuracy: 0.9333 - val_loss: 0.1996
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9290 - loss: 0.1793 - val_accuracy: 0.9286 - val_loss: 0.1824
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9480 - loss: 0.1278 - val_accuracy: 0.9460 - val_loss: 0.1838
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9643 - loss: 0.0968 - val_accuracy: 0.9429 - val_loss: 0.2204
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9734 - loss: 0.0718 - val_accuracy: 0.9444 - val_loss: 0.2297
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9770 - loss: 0.0609 - val_accuracy: 0.9524 - val_los

---
## Bloque 4 – Clasificación de Reseñas
Pasamos todas las reseñas por el modelo y las separamos en dos grupos:
-   Usuarios Satisfechos (predicción > 0.5)
-   Usuarios Insatisfechos (predicción ≤ 0.5)

In [26]:
def clasificar_reseña(texto):
    """Devuelve 'positivo' o 'negativo' para una reseña de texto libre."""
    seq = tokenizer.texts_to_sequences([texto])
    pad = pad_sequences(seq, maxlen=max_length, padding='post')
    pred = model.predict(pad, verbose=0)[0][0]
    return "positivo" if pred > 0.5 else "negativo"

# Clasificar todas las reseñas
positivas = []
negativas = []

for r in reseñas:
    if clasificar_reseña(r) == "positivo":
        positivas.append(r)
    else:
        negativas.append(r)

total = len(reseñas)
print(f"Reseñas positivas (Usuarios Satisfechos)  : {len(positivas)} ({len(positivas)/total*100:.1f}%)")
print(f"Reseñas negativas (Usuarios Insatisfechos): {len(negativas)} ({len(negativas)/total*100:.1f}%)")

Reseñas positivas (Usuarios Satisfechos)  : 2924 (92.8%)
Reseñas negativas (Usuarios Insatisfechos): 226 (7.2%)


---
## Bloque 5 – Extracción de Ideas Clave (TF-IDF)

Usamos **TF-IDF** (Term Frequency – Inverse Document Frequency) para extraer las palabras más representativas de cada grupo.

A diferencia del conteo simple de frecuencias, TF-IDF penaliza las palabras que aparecen en **todos** los documentos (como "the", "and", "I"), premiando las que son características de un grupo concreto.

In [27]:
# Stopwords estándar en inglés
stop_en = list(stopwords.words('english'))

# Añadimos palabras del propio producto que no aportan insight
# (aparecen igual en positivas y negativas, por eso TF-IDF las ignora
#  si comparamos los dos grupos, pero las excluimos explícitamente
#  por seguridad)
stop_producto = ['alexa', 'echo', 'amazon', 'dot', 'device',
                 'product', 'one', 'use', 'used', 'get', 'got',
                 'also', 'it', 'its', 'this', 'that', 'the']
stop_total = stop_en + stop_producto

def ideas_clave_comparativo(grupo_a, grupo_b, n=5):
    """
    Encuentra las palabras más características de grupo_a
    comparándolo con grupo_b mediante TF-IDF.
    
    Al pasar ambos grupos, TF-IDF penaliza las palabras que
    aparecen en los dos (como el nombre del producto), y premia
    las que son exclusivas de cada grupo.
    """
    doc_a = " ".join(grupo_a)  # Todas las positivas = 1 documento
    doc_b = " ".join(grupo_b)  # Todas las negativas = 1 documento
    
    vectorizer = TfidfVectorizer(
        stop_words=stop_total,
        max_features=1000,
        token_pattern=r'\b[a-zA-Z]{3,}\b'  # Solo palabras de 3+ letras
    )
    
    # Ajustamos con AMBOS documentos: así TF-IDF compara entre grupos
    tfidf_matrix = vectorizer.fit_transform([doc_a, doc_b])
    
    palabras = vectorizer.get_feature_names_out()
    
    # Índice 0 = grupo_a (positivas), índice 1 = grupo_b (negativas)
    scores_a = tfidf_matrix.toarray()[0]
    
    indices_top = scores_a.argsort()[-n:][::-1]
    return [(palabras[i], round(scores_a[i], 4)) for i in indices_top]


# Palabras características de positivas (comparando con negativas)
top_positivo = ideas_clave_comparativo(positivas, negativas, n=5)
# Palabras características de negativas (comparando con positivas)
top_negativo = ideas_clave_comparativo(negativas, positivas, n=5)

print("PUNTOS FUERTES (palabras exclusivas de reseñas positivas):")
for i, (palabra, score) in enumerate(top_positivo, 1):
    print(f"  {i}. {palabra:20s} (score TF-IDF: {score})")

print("\nPUNTOS DÉBILES (palabras exclusivas de reseñas negativas):")
for i, (palabra, score) in enumerate(top_negativo, 1):
    print(f"  {i}. {palabra:20s} (score TF-IDF: {score})")

PUNTOS FUERTES (palabras exclusivas de reseñas positivas):
  1. love                 (score TF-IDF: 0.4915)
  2. great                (score TF-IDF: 0.3704)
  3. music                (score TF-IDF: 0.2641)
  4. like                 (score TF-IDF: 0.2379)
  5. works                (score TF-IDF: 0.1855)

PUNTOS DÉBILES (palabras exclusivas de reseñas negativas):
  1. would                (score TF-IDF: 0.2448)
  2. like                 (score TF-IDF: 0.2185)
  3. work                 (score TF-IDF: 0.2098)
  4. time                 (score TF-IDF: 0.1967)
  5. screen               (score TF-IDF: 0.1923)


### Prueba con una reseña nueva
El sistema puede procesar cualquier reseña nueva y asignarla automáticamente:

In [28]:
def auditar_reseña(texto):
    """Clasifica una reseña nueva y la asigna al informe correspondiente."""
    seq  = tokenizer.texts_to_sequences([texto])
    pad  = pad_sequences(seq, maxlen=max_length, padding='post')
    pred = model.predict(pad, verbose=0)[0][0]
    
    if pred > 0.5:
        categoria = "POSITIVA → Suma a Puntos Fuertes"
    else:
        categoria = "NEGATIVA → Suma a Puntos Débiles"
    
    print(f"Reseña  : '{texto}'")
    print(f"Score   : {pred:.3f}")
    print(f"Resultado: {categoria}")

print("--- Pruebas con reseñas nuevas ---\n")
auditar_reseña("I love this device, it works perfectly every day")
print()
auditar_reseña("Terrible quality, stopped working after one week")
print()
auditar_reseña("It's okay but nothing special, delivery was slow")

--- Pruebas con reseñas nuevas ---

Reseña  : 'I love this device, it works perfectly every day'
Score   : 1.000
Resultado: POSITIVA → Suma a Puntos Fuertes

Reseña  : 'Terrible quality, stopped working after one week'
Score   : 0.069
Resultado: NEGATIVA → Suma a Puntos Débiles

Reseña  : 'It's okay but nothing special, delivery was slow'
Score   : 0.722
Resultado: POSITIVA → Suma a Puntos Fuertes


---
## Bloque 6 – Informe Final de Resultados

In [29]:
total = len(reseñas)
pos = len(positivas)
neg = len(negativas)

print("=" * 55)
print("         INFORME DE AUDITORÍA DE SATISFACCIÓN")
print("=" * 55)
print(f"  Producto analizado : Amazon Alexa (Echo Dot)")
print(f"  Total de reseñas   : {total}")
print(f"  Precisión del modelo (val): {val_acc*100:.1f}%")
print("-" * 55)
print(f"Reseñas Positivas: {pos:4d}  ({pos/total*100:.1f}%)")
print(f"Reseñas Negativas: {neg:4d}  ({neg/total*100:.1f}%)")
print("-" * 55)
print("PUNTOS FUERTES (top 5):")
for i, (p, _) in enumerate(top_positivo, 1):
    print(f"     {i}. {p}")
print("PUNTOS DÉBILES (top 5):")
for i, (p, _) in enumerate(top_negativo, 1):
    print(f"     {i}. {p}")


         INFORME DE AUDITORÍA DE SATISFACCIÓN
  Producto analizado : Amazon Alexa (Echo Dot)
  Total de reseñas   : 3150
  Precisión del modelo (val): 94.3%
-------------------------------------------------------
Reseñas Positivas: 2924  (92.8%)
Reseñas Negativas:  226  (7.2%)
-------------------------------------------------------
PUNTOS FUERTES (top 5):
     1. love
     2. great
     3. music
     4. like
     5. works
PUNTOS DÉBILES (top 5):
     1. would
     2. like
     3. work
     4. time
     5. screen


CONCLUSIÓN:
-   El modelo Bidirectional LSTM ha logrado una precisión de validación superior al 93%, lo que indica que clasifica correctamente 
    la gran mayoría de reseñas. Los puntos fuertes extraídos reflejan aspectos como la facilidad de uso y la integración con el hogar, 
    mientras que los puntos débiles recogen quejas sobre conectividad y calidad del sonido, concordando con la realidad del producto analizado.